<a href="https://colab.research.google.com/github/engMohamedAbdAlslam/DRP_segmentation/blob/copilot%2Fdevelop-preprocessing-pipeline/notebooks/03_binary_classification_filter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 03 — Binary Classification Filter (DR vs No-DR)
**Dataset:** Diabetic Retinopathy Resized (`tanlikesmath/diabetic-retinopathy-resized`)
**Goal:** Download via KaggleHub, preprocess fundus images with DR grade labels, save as `.npz`, analyze class distribution.
> Run on **Google Colab** with T4 GPU. Set your Kaggle credentials in Colab Secrets before running.

## 1. Colab Repo Setup

In [ ]:
import os
from pathlib import Path

repo_path = Path('/content/DRP_segmentation')
if not repo_path.exists():
    !git clone https://github.com/engMohamedAbdAlslam/DRP_segmentation.git /content/DRP_segmentation

%cd /content/DRP_segmentation
!git checkout copilot/develop-preprocessing-pipeline
!git pull origin copilot/develop-preprocessing-pipeline
print('Repo ready at', Path.cwd())

## 2. Install Dependencies

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', '-q', 'install',
                'kagglehub', 'opencv-python-headless', 'tqdm', 'matplotlib', 'scikit-learn', 'numpy', 'pandas'],
               check=True)
print('Dependencies ready.')

## 3. Kaggle Authentication

In [ ]:
from google.colab import userdata
import os, json

kaggle_token = json.loads(userdata.get('KAGGLE_TOKEN'))
os.environ['KAGGLE_USERNAME'] = kaggle_token['username']
os.environ['KAGGLE_KEY']      = kaggle_token['key']
print('Kaggle credentials ready!')

## 4. Imports & Path Setup

In [ ]:
import sys, shutil
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm import tqdm
from sklearn.model_selection import train_test_split

print('numpy', np.__version__)
print('opencv', cv2.__version__)

repo_root = Path('/content/DRP_segmentation')
if str(repo_root / 'src') not in sys.path:
    sys.path.insert(0, str(repo_root / 'src'))

from engine.image_preprocessing import PreprocessConfig, preprocess_fundus_image, save_preprocessed

data_dir      = repo_root / 'data'
raw_dir       = data_dir / 'raw'
processed_dir = data_dir / 'processed'
raw_dir.mkdir(parents=True, exist_ok=True)
processed_dir.mkdir(parents=True, exist_ok=True)
print('Paths ready.')

## 5. Download DR Resized Dataset

In [ ]:
import kagglehub

DATASET_SLUG = 'tanlikesmath/diabetic-retinopathy-resized'
DATASET_NAME = 'dr_resized'
dr_root      = raw_dir / 'dr_resized'
dr_root.mkdir(parents=True, exist_ok=True)

already_downloaded = any(dr_root.rglob('*.png')) or any(dr_root.rglob('*.jpeg')) or any(dr_root.rglob('*.csv'))
if not already_downloaded:
    print(f'Downloading {DATASET_SLUG} ...')
    download_path = Path(kagglehub.dataset_download(DATASET_SLUG))
    shutil.copytree(str(download_path), str(dr_root), dirs_exist_ok=True)
    print('Download complete:', dr_root)
else:
    print('Dataset already present at:', dr_root)

print(f'Total files: {len(list(dr_root.rglob("*")))}') 

## 6. Load Labels & Index Images

In [ ]:
IMAGE_EXTS = {'.png', '.jpg', '.jpeg'}
DR_GRADE_NAMES = {0: 'No DR', 1: 'Mild', 2: 'Moderate', 3: 'Severe', 4: 'Proliferative'}

csv_files = list(dr_root.rglob('*.csv'))
df = None
if csv_files:
    label_csv = csv_files[0]
    print(f'Found label file: {label_csv}')
    raw_df = pd.read_csv(label_csv)
    print('Columns:', list(raw_df.columns))
    print(raw_df.head())
    img_col = next((c for c in raw_df.columns if 'image' in c.lower() or 'file' in c.lower() or 'name' in c.lower()), raw_df.columns[0])
    lbl_col = next((c for c in raw_df.columns if 'level' in c.lower() or 'label' in c.lower() or 'grade' in c.lower() or 'class' in c.lower()), raw_df.columns[1])
    all_imgs = list(dr_root.rglob('*'))
    img_map  = {p.stem: p for p in all_imgs if p.suffix.lower() in IMAGE_EXTS}
    rows = []
    for _, row in raw_df.iterrows():
        stem = Path(str(row[img_col])).stem
        img_path = img_map.get(stem)
        if img_path:
            rows.append({'image_path': img_path, 'dr_grade': int(row[lbl_col]), 'binary_label': 0 if int(row[lbl_col]) == 0 else 1})
    df = pd.DataFrame(rows)
else:
    print('No CSV found. Inferring labels from folder names...')
    rows = []
    for img_path in dr_root.rglob('*'):
        if img_path.suffix.lower() not in IMAGE_EXTS: continue
        try: label = int(img_path.parent.name)
        except ValueError: label = -1
        rows.append({'image_path': img_path, 'dr_grade': label, 'binary_label': 0 if label == 0 else 1})
    df = pd.DataFrame(rows)

print(f'\nTotal images indexed: {len(df)}')
print('\nDR Grade distribution:')
print(df['dr_grade'].map(DR_GRADE_NAMES).value_counts().sort_index())

## 7. Train / Val / Test Split (70/15/15) — Stratified

In [ ]:
if df.empty:
    raise RuntimeError('No images found — check dataset download.')

stratify_col = df['binary_label'] if df['binary_label'].nunique() > 1 else None
train_df, temp_df = train_test_split(df, test_size=0.30, random_state=42, stratify=stratify_col)
s2 = temp_df['binary_label'] if temp_df['binary_label'].nunique() > 1 else None
val_df, test_df = train_test_split(temp_df, test_size=0.50, random_state=42, stratify=s2)

splits = {'train': train_df, 'val': val_df, 'test': test_df}
for name, sdf in splits.items():
    print(f'{name}: {len(sdf)} images | binary dist: {sdf["binary_label"].value_counts().to_dict()}')

## 8. Batch Preprocessing & Save as .npz

In [ ]:
config = PreprocessConfig(target_size=(512, 512), normalization='zero_one')
errors = []

for split_name, split_df in splits.items():
    out_dir = processed_dir / DATASET_NAME / split_name
    out_dir.mkdir(parents=True, exist_ok=True)
    print(f'\nProcessing {split_name} ({len(split_df)} images)...')
    for _, row in tqdm(split_df.iterrows(), total=len(split_df), desc=split_name):
        try:
            result   = preprocess_fundus_image(row['image_path'], mask=None, config=config)
            out_file = out_dir / (row['image_path'].stem + f'_grade{row["dr_grade"]}.npz')
            save_preprocessed(result, out_file)
        except Exception as e:
            errors.append({'file': str(row['image_path']), 'error': str(e)})

print(f'\nDone. Errors: {len(errors)}')
if errors:
    print(pd.DataFrame(errors))

## 9. Dataset Statistics & Class Distribution

In [ ]:
print('=== Processed .npz Counts ===')
for split_name in splits:
    count = len(list((processed_dir / DATASET_NAME / split_name).rglob('*.npz')))
    print(f'  {split_name}: {count} files')

grade_names = {0: 'No DR', 1: 'Mild', 2: 'Moderate', 3: 'Severe', 4: 'Proliferative'}
counts  = df['dr_grade'].value_counts().sort_index()
labels  = [grade_names.get(i, f'Grade {i}') for i in counts.index]
colors  = ['#2ecc71', '#f39c12', '#e67e22', '#e74c3c', '#8e44ad'][:len(counts)]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.bar(labels, counts.values, color=colors, edgecolor='black', linewidth=0.7)
ax1.set_title('DR Grade Distribution (5-class)', fontsize=12, fontweight='bold')
ax1.set_xlabel('DR Grade'); ax1.set_ylabel('Number of Images')
for i, v in enumerate(counts.values): ax1.text(i, v + max(counts.values)*0.01, str(v), ha='center', fontsize=9)

binary_counts = df['binary_label'].value_counts().sort_index()
ax2.pie(binary_counts.values, labels=['No DR (Grade 0)', 'DR (Grade 1-4)'],
        autopct='%1.1f%%', colors=['#2ecc71', '#e74c3c'], startangle=90, textprops={'fontsize': 11})
ax2.set_title('Binary Classification Distribution', fontsize=12, fontweight='bold')

plt.suptitle('DR Resized Dataset — Class Distribution', fontsize=13)
plt.tight_layout()
plt.show()
print(f'\nClass imbalance ratio (DR/No-DR): {binary_counts.get(1,0)/max(binary_counts.get(0,1),1):.2f}')

## 10. Visualization — Sample Images per DR Grade

In [ ]:
unique_labels = sorted(df['dr_grade'].unique())
n_cols = min(5, len(unique_labels))
fig, axes = plt.subplots(1, n_cols, figsize=(4 * n_cols, 4))
if n_cols == 1: axes = [axes]

for i, lbl in enumerate(unique_labels):
    subset = df[df['dr_grade'] == lbl]
    sample_row = subset.sample(1, random_state=42).iloc[0]
    img_bgr = cv2.imread(str(sample_row['image_path']))
    if img_bgr is not None:
        axes[i].imshow(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
    axes[i].set_title(f'Grade {lbl}: {DR_GRADE_NAMES.get(lbl, "?")}\n(n={len(subset)})', fontsize=9)
    axes[i].axis('off')

plt.suptitle('DR Resized — One Sample per Grade', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 11. Save Processed Files to Google Drive
> Run once after Cell 8. Saved to: `My Drive/DRP_processed/dr_resized/`

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

drive_dest = Path('/content/drive/MyDrive/DRP_processed')
drive_dest.mkdir(parents=True, exist_ok=True)

src = processed_dir / DATASET_NAME
if src.exists():
    shutil.copytree(str(src), str(drive_dest / DATASET_NAME), dirs_exist_ok=True)
    npz_count = len(list((drive_dest / DATASET_NAME).rglob('*.npz')))
    print(f'Saved {npz_count} .npz files → {drive_dest / DATASET_NAME}')
else:
    print('No processed files found. Run Cell 8 first.')

## Next Steps — Model Training
- Load `.npz` from `data/processed/dr_resized/`
- Build a **binary DR classifier**: EfficientNet-B4 or ResNet-50
- Address class imbalance with: **weighted cross-entropy**, **focal loss**, or **oversampling**
- Use this as a **pre-filter** before running the full grading pipeline from Notebook 01
- Metrics: **AUC-ROC**, **F1-score**, **Sensitivity**, **Specificity** (clinical priority: high sensitivity)